# Coronal Waves and Oscillations — Implementation
# 코로나 파동과 진동 — 구현

**Paper**: Nakariakov, V. M. & Verwichte, E. (2005), *Living Rev. Solar Phys.*, **2**, 3

이 노트북은 논문의 핵심 개념을 코드로 구현합니다:
This notebook implements the key concepts from the paper:

1. **Part 1**: MHD 분산 관계 — Edwin & Roberts (1983) dispersion diagram 재현
2. **Part 2**: Kink 진동과 코로나 자기장 추정 — MHD coronal seismology
3. **Part 3**: 공진 흡수 vs 위상 혼합 — 감쇠 메커니즘 비교
4. **Part 4**: Sausage 모드 존재 조건과 cutoff
5. **Part 5**: Alfvén 위상 혼합 시뮬레이션
6. **Part 6**: 전파하는 느린 파동의 Burgers 방정식 모델링
7. **Part 7**: Fast wave train의 분산 진화와 "tadpole" 시그니처

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import iv, kv, ivp, kvp  # Modified Bessel functions
from scipy.optimize import brentq
from scipy.signal import morlet2
from scipy.ndimage import gaussian_filter1d

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Part 1: MHD Dispersion Relation — Edwin & Roberts (1983) Diagram
## 파트 1: MHD 분산 관계 — Edwin & Roberts (1983) 다이어그램

자기 실린더에서 MHD 파동의 분산 관계(Eq. 7)를 수치적으로 풀어 Figure 3을 재현합니다.
We numerically solve the dispersion relation (Eq. 7) for MHD waves in a magnetic cylinder to reproduce Figure 3.

분산 관계 / Dispersion relation:

$$\rho_e(\omega^2 - k_z^2 C_{Ae}^2)\kappa_0 \frac{I_m'(\kappa_0 a)}{I_m(\kappa_0 a)} + \rho_0(k_z^2 C_{A0}^2 - \omega^2)\kappa_e \frac{K_m'(\kappa_e a)}{K_m(\kappa_e a)} = 0$$

여기서 / where:
$$\kappa^2 = -\frac{(\omega^2 - C_s^2 k_z^2)(\omega^2 - C_A^2 k_z^2)}{(C_s^2 + C_A^2)(\omega^2 - C_T^2 k_z^2)}$$

In [ ]:
def kappa_sq(omega, kz, Cs, CA):
    """Compute kappa^2 (Eq. 5) for transverse wavenumber.

    Args:
        omega: Angular frequency.
        kz: Longitudinal wavenumber.
        Cs: Sound speed.
        CA: Alfvén speed.

    Returns:
        kappa^2 value.
    """
    CT = Cs * CA / np.sqrt(Cs**2 + CA**2)
    num = (omega**2 - Cs**2 * kz**2) * (omega**2 - CA**2 * kz**2)
    den = (Cs**2 + CA**2) * (omega**2 - CT**2 * kz**2)
    return -num / den


def dispersion_func(vph, kz_a, m, Cs0, CA0, Cse, CAe, rho0, rhoe):
    """Evaluate the dispersion relation (Eq. 7) for root-finding.

    Args:
        vph: Phase speed omega/kz to solve for.
        kz_a: Dimensionless wavenumber kz * a.
        m: Azimuthal mode number.
        Cs0: Internal sound speed.
        CA0: Internal Alfvén speed.
        Cse: External sound speed.
        CAe: External Alfvén speed.
        rho0: Internal density.
        rhoe: External density.

    Returns:
        Residual of the dispersion relation.
    """
    kz = kz_a  # a = 1
    omega = vph * kz

    k0_sq = kappa_sq(omega, kz, Cs0, CA0)
    ke_sq = kappa_sq(omega, kz, Cse, CAe)

    # Both must be positive for trapped body modes
    if k0_sq <= 0 or ke_sq <= 0:
        return np.nan

    k0 = np.sqrt(k0_sq)
    ke = np.sqrt(ke_sq)

    k0a = k0  # a = 1
    kea = ke

    # Use derivatives: I_m'(x) = ivp(m, x), K_m'(x) = kvp(m, x)
    try:
        term1 = rhoe * (omega**2 - kz**2 * CAe**2) * k0 * ivp(m, k0a) / iv(m, k0a)
        term2 = rho0 * (kz**2 * CA0**2 - omega**2) * ke * kvp(m, kea) / kv(m, kea)
    except (FloatingPointError, ZeroDivisionError):
        return np.nan

    return term1 + term2


def solve_dispersion(m, Cs0, CA0, Cse, CAe, rho0, rhoe, kz_a_range,
                     vph_bounds, n_kz=200):
    """Solve the dispersion relation for a given mode number.

    Args:
        m: Azimuthal mode number.
        Cs0, CA0, Cse, CAe: Internal/external sound/Alfvén speeds.
        rho0, rhoe: Internal/external densities.
        kz_a_range: Tuple (min, max) for kz*a.
        vph_bounds: List of (vph_min, vph_max) tuples for each branch.
        n_kz: Number of kz points.

    Returns:
        List of (kz_a_array, vph_array) tuples for each branch found.
    """
    kz_a_vals = np.linspace(kz_a_range[0], kz_a_range[1], n_kz)
    branches = []

    for vmin, vmax in vph_bounds:
        kz_list = []
        vph_list = []
        for kza in kz_a_vals:
            if kza < 0.01:
                continue
            # Scan for sign changes
            test_v = np.linspace(vmin + 1e-6, vmax - 1e-6, 500)
            vals = []
            for v in test_v:
                val = dispersion_func(v, kza, m, Cs0, CA0, Cse, CAe, rho0, rhoe)
                vals.append(val if not np.isnan(val) else None)

            # Find roots via sign changes
            roots = []
            for i in range(len(vals) - 1):
                if vals[i] is not None and vals[i + 1] is not None:
                    if vals[i] * vals[i + 1] < 0:
                        try:
                            root = brentq(
                                dispersion_func, test_v[i], test_v[i + 1],
                                args=(kza, m, Cs0, CA0, Cse, CAe, rho0, rhoe)
                            )
                            roots.append(root)
                        except (ValueError, RuntimeError):
                            pass

            for r in roots:
                kz_list.append(kza)
                vph_list.append(r)

        if kz_list:
            branches.append((np.array(kz_list), np.array(vph_list)))

    return branches

In [ ]:
# Parameters matching Figure 3: CA0 = 2Cs0, CAe = 5Cs0, Cse = 0.5Cs0
# Normalize all speeds to Cs0 = 1
Cs0 = 1.0
CA0 = 2.0
Cse = 0.5
CAe = 5.0

CT0 = Cs0 * CA0 / np.sqrt(Cs0**2 + CA0**2)
CTe = Cse * CAe / np.sqrt(Cse**2 + CAe**2)

# Density from pressure balance: p0 + B0^2/2mu0 = pe + Be^2/2mu0
# Using CA = B/sqrt(mu0*rho), Cs^2 = gamma*p/rho
# rho0/rhoe determined by speed ratios
rho0 = 1.0
rhoe = rho0 * CA0**2 / CAe**2  # From magnetic pressure balance (simplified)

# Kink speed
CK = np.sqrt((rho0 * CA0**2 + rhoe * CAe**2) / (rho0 + rhoe))

print("=== Characteristic Speeds (normalized to Cs0) ===")
print(f"Cs0  = {Cs0:.2f},  CA0  = {CA0:.2f},  CT0  = {CT0:.4f}")
print(f"Cse  = {Cse:.2f},  CAe  = {CAe:.2f},  CTe  = {CTe:.4f}")
print(f"CK   = {CK:.4f}")
print(f"rho0 = {rho0:.2f},  rhoe = {rhoe:.4f}")

# Solve for m = 0, 1, 2, 3 modes in the fast band (CA0 < vph < CAe)
fig, ax = plt.subplots(figsize=(10, 8))

styles = ["-", ":", "--", "-."]
labels_m = ["m=0 (sausage)", "m=1 (kink)", "m=2 (flute)", "m=3 (flute)"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for m_val in range(4):
    branches = solve_dispersion(
        m=m_val, Cs0=Cs0, CA0=CA0, Cse=Cse, CAe=CAe,
        rho0=rho0, rhoe=rhoe,
        kz_a_range=(0.05, 5.0),
        vph_bounds=[(CA0 + 0.01, CAe - 0.01)],  # Fast band
        n_kz=150,
    )
    for i, (kz_arr, vph_arr) in enumerate(branches):
        label = labels_m[m_val] if i == 0 else None
        ax.scatter(kz_arr, vph_arr, s=1, color=colors[m_val], label=label)

# Also solve slow band (CT0 < vph < Cs0)
for m_val in range(4):
    branches = solve_dispersion(
        m=m_val, Cs0=Cs0, CA0=CA0, Cse=Cse, CAe=CAe,
        rho0=rho0, rhoe=rhoe,
        kz_a_range=(0.05, 5.0),
        vph_bounds=[(CT0 + 0.001, Cs0 - 0.001)],  # Slow band
        n_kz=150,
    )
    for kz_arr, vph_arr in branches:
        ax.scatter(kz_arr, vph_arr, s=1, color=colors[m_val])

# Alfvén wave line
ax.axhline(y=CA0, color="black", linestyle="--", alpha=0.5, label=f"$C_{{A0}}$ = {CA0}")
ax.axhline(y=CAe, color="gray", linestyle="--", alpha=0.5, label=f"$C_{{Ae}}$ = {CAe}")
ax.axhline(y=CK, color="purple", linestyle=":", alpha=0.5, label=f"$C_K$ = {CK:.2f}")
ax.axhline(y=Cs0, color="green", linestyle="--", alpha=0.3, label=f"$C_{{s0}}$ = {Cs0}")
ax.axhline(y=CT0, color="brown", linestyle="--", alpha=0.3, label=f"$C_{{T0}}$ = {CT0:.2f}")
ax.axhline(y=Cse, color="olive", linestyle="--", alpha=0.3, label=f"$C_{{se}}$ = {Cse}")
ax.axhline(y=CTe, color="pink", linestyle="--", alpha=0.3, label=f"$C_{{Te}}$ = {CTe:.2f}")

ax.set_xlabel(r"$k_z a$", fontsize=14)
ax.set_ylabel(r"$\omega / k_z$ (phase speed)", fontsize=14)
ax.set_title("MHD Dispersion Diagram — Magnetic Cylinder\n"
             "MHD 분산 다이어그램 — 자기 실린더 (cf. Figure 3)", fontsize=14)
ax.set_xlim(0, 5)
ax.set_ylim(0, 5.5)
ax.legend(fontsize=9, loc="right")
plt.tight_layout()
plt.show()

## Part 2: Kink Oscillations & Coronal Magnetic Field Estimation
## 파트 2: Kink 진동과 코로나 자기장 추정

1998년 7월 14일 TRACE 관측의 kink 진동 데이터를 재현하고, Eq. (32)를 사용하여 코로나 자기장을 추정합니다.
We reproduce the kink oscillation data from the 14 July 1998 TRACE observation and estimate the coronal magnetic field using Eq. (32).

$$B_0 \approx 1.02 \times 10^{-12} \frac{d\sqrt{\mu n_0}\sqrt{1 + n_e/n_0}}{P}$$

In [ ]:
# --- Reproduce Figure 10: Damped kink oscillation ---
# Best-fit parameters from Nakariakov et al. (1999)
A = 2030  # km, amplitude
omega_obs = 1.47  # rad/min
lam = 0.069  # min^-1, decay rate
phi = 0.0  # phase

t = np.linspace(0, 27, 500)  # minutes
displacement = A * np.sin(omega_obs * t + phi) * np.exp(-lam * t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: damped oscillation
ax1.plot(t, displacement, "b-", lw=2)
ax1.fill_between(t, -A * np.exp(-lam * t), A * np.exp(-lam * t),
                  alpha=0.1, color="blue")
ax1.plot(t, A * np.exp(-lam * t), "r--", alpha=0.5, label="Envelope")
ax1.plot(t, -A * np.exp(-lam * t), "r--", alpha=0.5)
ax1.set_xlabel("Time (min)", fontsize=12)
ax1.set_ylabel("Displacement (km)", fontsize=12)
ax1.set_title("Damped Kink Oscillation (cf. Figure 10)\n"
              "감쇠 kink 진동", fontsize=12)
ax1.legend()

# --- Coronal magnetic field estimation (Eq. 32) ---
# Parameters
d = 83e6  # m (footpoint distance = 83 Mm)
mu_particle = 1.27  # effective particle mass in units of proton mass
m_p = 1.67e-27  # kg (proton mass)
mu_0_SI = 4 * np.pi * 1e-7  # H/m

P_obs = 256  # s (oscillation period)

# Scan over density n0
n0_range = np.linspace(1e15, 6e15, 200)  # m^-3 (= 10^9 to 6×10^9 cm^-3)
ne_over_n0 = 0.1  # density ratio

B0_values = 1.02e-12 * d * np.sqrt(mu_particle * n0_range) * np.sqrt(1 + ne_over_n0) / P_obs

# Convert to Gauss (1 T = 10^4 G), and n0 to cm^-3
B0_G = B0_values * 1e4  # Already in Gauss from the formula
n0_cm3 = n0_range * 1e-6  # m^-3 to cm^-3

ax2.plot(n0_cm3, B0_G, "b-", lw=2, label=r"$\rho_e/\rho_0$ = 0.1")

# Error bounds from kink speed uncertainty (1020 ± 132 km/s)
CK_center = 1020e3  # m/s
CK_upper = 1152e3
CK_lower = 888e3

# B0 = sqrt(mu0 * rho0) * CA0
# CA0 = CK * sqrt((1 + rho_e/rho_0) / 2) for low-beta
for CK_val, ls, label in [(CK_upper, "--", "Upper bound"),
                           (CK_lower, "--", "Lower bound")]:
    CA0_val = CK_val * np.sqrt((1 + ne_over_n0) / 2)
    B0_bound = np.sqrt(mu_0_SI * mu_particle * m_p * n0_range) * CA0_val * 1e4
    ax2.plot(n0_cm3, B0_bound, ls=ls, color="gray", alpha=0.7, label=label)

# Mark typical density range
ax2.axvline(x=1e9, color="green", ls=":", alpha=0.5)
ax2.axvline(x=5e9, color="green", ls=":", alpha=0.5)
ax2.axvspan(1e9, 5e9, alpha=0.05, color="green", label="Typical density range")

ax2.set_xlabel("Number density $n_0$ (cm$^{-3}$)", fontsize=12)
ax2.set_ylabel("Magnetic field $B_0$ (G)", fontsize=12)
ax2.set_title("Coronal Magnetic Field from Kink Oscillations (cf. Fig 12)\n"
              "Kink 진동에서 추정한 코로나 자기장", fontsize=12)
ax2.legend(fontsize=9)
ax2.set_ylim(0, 25)

plt.tight_layout()
plt.show()

# Print key results
print("=== Coronal Seismology Results ===")
print(f"Kink speed: CK = {CK_center/1e3:.0f} ± 132 km/s")
CA_est = CK_center * np.sqrt((1 + ne_over_n0) / 2)
print(f"Alfvén speed (ρe/ρ0=0.1): CA0 = {CA_est/1e3:.0f} km/s")
n0_typical = 3e15  # m^-3
B0_est = np.sqrt(mu_0_SI * mu_particle * m_p * n0_typical) * CA_est * 1e4
print(f"B0 (n0 = 3×10^9 cm^-3): {B0_est:.1f} G")
print(f"Period: P = {P_obs} s = {P_obs/60:.1f} min")
print(f"Decay time: τ = {1/lam:.1f} min")

## Part 3: Damping Mechanisms — Resonant Absorption vs Phase Mixing
## 파트 3: 감쇠 메커니즘 — 공진 흡수 vs 위상 혼합

관측된 감쇠 시간과 이론적 예측을 비교합니다. 네 가지 감쇠 메커니즘의 스케일링 법칙:
We compare observed decay times with theoretical predictions. Scaling laws for four damping mechanisms:

- **Resonant absorption** (Eq. 34): $\tau/P = (2/\pi)(\ell/a)^{-1}(\rho_0+\rho_e)/(\rho_0-\rho_e)$, i.e. $\tau \propto P$
- **Phase mixing** (Eq. 35): $\tau/P = (3/4\pi^2)^{1/3}(\ell/L)^{2/3}\text{Re}^{1/3}$, i.e. $\tau \propto P^{2/3}$
- **Coronal leakage** (Eq. 36): $\tau \propto L^2 P$
- **Footpoint leakage** (Eq. 37): $\tau/P = (1/4\pi^2)(h/L)^{-1}$

In [ ]:
# --- Compare damping mechanisms ---

# Resonant absorption: tau/P as function of l/a for different density ratios
l_over_a = np.linspace(0.01, 0.5, 200)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Resonant absorption - tau/P vs l/a
ax = axes[0]
for rho_ratio, color, label in [(2, "blue", r"$\rho_0/\rho_e$ = 2"),
                                  (5, "red", r"$\rho_0/\rho_e$ = 5"),
                                  (10, "green", r"$\rho_0/\rho_e$ = 10"),
                                  (20, "purple", r"$\rho_0/\rho_e$ = 20")]:
    tau_over_P = (2 / np.pi) * (1 / l_over_a) * (rho_ratio + 1) / (rho_ratio - 1)
    ax.plot(l_over_a, tau_over_P, color=color, lw=2, label=label)

ax.axhline(y=14.5 / 4.3, color="black", ls="--", alpha=0.5,
           label=f"Observed τ/P = {14.5/4.3:.1f}")
ax.set_xlabel(r"$\ell / a$ (boundary layer width)", fontsize=11)
ax.set_ylabel(r"$\tau / P$", fontsize=11)
ax.set_title("Resonant Absorption (Eq. 34)\n공진 흡수", fontsize=11)
ax.set_ylim(0, 20)
ax.legend(fontsize=9)

# Panel 2: Scaling laws comparison - tau vs P
ax = axes[1]
P_range = np.linspace(50, 1000, 200)  # seconds

# Observed data points (approximate from Figure 13)
P_obs_data = np.array([256, 396, 185, 234, 522, 236, 272, 316])  # seconds
tau_obs_data = np.array([870, 660, 200, 400, 1200, 714, 849, 500])  # seconds

ax.scatter(P_obs_data, tau_obs_data, s=60, c="black", zorder=5,
           label="Observed events")

# Resonant absorption: tau = c1 * P
c1 = np.median(tau_obs_data / P_obs_data)
ax.plot(P_range, c1 * P_range, "b-", lw=2, label=f"Res. absorption: τ ∝ P")

# Phase mixing: tau = c2 * P^(4/3) (modified, Eq. 38)
c2 = np.median(tau_obs_data / P_obs_data**(4/3))
ax.plot(P_range, c2 * P_range**(4/3), "r--", lw=2,
        label=r"Phase mixing: τ ∝ P$^{4/3}$")

# Footpoint leakage: tau = c3 * P^2
c3 = np.median(tau_obs_data / P_obs_data**2)
ax.plot(P_range, c3 * P_range**2, "g:", lw=2,
        label=r"Footpoint leak: τ ∝ P$^2$")

ax.set_xlabel("Period P (s)", fontsize=11)
ax.set_ylabel("Decay time τ (s)", fontsize=11)
ax.set_title("Scaling Laws Comparison (cf. Fig. 13)\n스케일링 법칙 비교", fontsize=11)
ax.set_xlim(50, 1000)
ax.set_ylim(10, 20000)
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend(fontsize=8)

# Panel 3: Inferred l/a from observations
ax = axes[2]
rho0_over_rhoe_range = np.linspace(2, 20, 100)
observed_tau_P = 14.5 / 4.3  # from Nakariakov et al. (1999)

l_a_inferred = (2 / np.pi) * (rho0_over_rhoe_range + 1) / \
    (rho0_over_rhoe_range - 1) / observed_tau_P

ax.plot(rho0_over_rhoe_range, l_a_inferred, "b-", lw=2)
ax.axhline(y=0.23, color="red", ls="--", alpha=0.5, label=r"$\ell/a$ = 0.23")
ax.set_xlabel(r"Density ratio $\rho_0 / \rho_e$", fontsize=11)
ax.set_ylabel(r"Inferred $\ell / a$", fontsize=11)
ax.set_title("Inferred Boundary Layer Width\n추론된 경계층 두께", fontsize=11)
ax.legend()

plt.tight_layout()
plt.show()

## Part 4: Sausage Mode Existence Condition & Cutoff
## 파트 4: Sausage 모드 존재 조건과 Cutoff

Global sausage mode가 존재하려면 루프가 충분히 두껍고 밀도가 높아야 합니다 (Eq. 47):
For the global sausage mode to exist, the loop must be sufficiently thick and dense (Eq. 47):

$$\frac{L}{2a} < 0.65\sqrt{\frac{\rho_0}{\rho_e}}$$

In [ ]:
# --- Sausage mode existence diagram ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: L/2a threshold vs density ratio
rho_ratio = np.linspace(1.5, 100, 300)
L_over_2a_max = 0.65 * np.sqrt(rho_ratio)

ax1.plot(rho_ratio, L_over_2a_max, "b-", lw=2)
ax1.fill_between(rho_ratio, 0, L_over_2a_max, alpha=0.15, color="blue",
                  label="Global sausage mode EXISTS\n전역 소시지 모드 존재")
ax1.fill_between(rho_ratio, L_over_2a_max, 10, alpha=0.1, color="red",
                  label="NO global sausage mode\n전역 소시지 모드 없음")

# Mark specific cases
# Typical active region loop: L ~ 200 Mm, 2a ~ 2 Mm -> L/2a = 100
ax1.plot(5, 100/2, "rv", ms=12, label="Typical AR loop\n(L/2a~50, ρ₀/ρₑ~5)")
# Flaring loop (Nakariakov 2003): L = 25 Mm, 2a = 6 Mm -> L/2a ≈ 4.2
ax1.plot(20, 25/6, "g^", ms=12, label="Flaring loop\n(L/2a≈4, ρ₀/ρₑ~20)")

ax1.set_xlabel(r"Density ratio $\rho_0 / \rho_e$", fontsize=12)
ax1.set_ylabel(r"$L / 2a$ (aspect ratio)", fontsize=12)
ax1.set_title("Sausage Mode Existence Condition (Eq. 47)\n소시지 모드 존재 조건",
              fontsize=12)
ax1.set_xlim(1, 100)
ax1.set_ylim(0, 10)
ax1.legend(fontsize=9, loc="upper left")

# Panel 2: Sausage mode period vs loop parameters
ax2_title = "Global Sausage Mode Period\n전역 소시지 모드 주기"

L_vals = np.array([15, 20, 25, 30, 40])  # Mm
CA0_vals = np.linspace(500, 2000, 200)  # km/s

for L_Mm in L_vals:
    # P_GSM = 2L / Cph, where CA0 < Cph < CAe
    # For rough estimate, Cph ≈ CA0 (lower bound for period)
    P_lower = 2 * L_Mm * 1e3 / CA0_vals  # in seconds
    ax2.plot(CA0_vals, P_lower, lw=2, label=f"L = {L_Mm} Mm")

ax2.axhspan(14, 17, alpha=0.2, color="yellow",
            label="Observed 14-17 s pulsations")
ax2.set_xlabel(r"Internal Alfvén speed $C_{A0}$ (km/s)", fontsize=12)
ax2.set_ylabel("Period $P_{GSM}$ (s)", fontsize=12)
ax2.set_title(ax2_title, fontsize=12)
ax2.set_ylim(0, 100)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Part 5: Alfvén Wave Phase Mixing Simulation
## 파트 5: Alfvén 파동 위상 혼합 시뮬레이션

비균질 플라즈마에서 Alfvén 파동의 위상 혼합을 시뮬레이션합니다 (§2.2.2).
We simulate phase mixing of Alfvén waves in inhomogeneous plasma (§2.2.2).

$$\left(\frac{\partial^2}{\partial t^2} - C_A^2(x)\frac{\partial^2}{\partial z^2}\right)V_y = 0$$

해 / Solution: $V_y = \Psi(x) f(z \mp C_A(x) t)$

In [ ]:
# --- Phase mixing simulation ---
# Alfvén speed profile: C_A(x) varies across the magnetic field
x = np.linspace(-3, 3, 400)  # transverse direction
z = 0.0  # fixed position along field

# Alfvén speed profile with gradient
CA_profile = 1.0 + 0.5 * np.tanh(x)  # smooth transition

# Standing Alfvén wave at different times
kz = 2 * np.pi  # wavenumber along field
times = [0, 1, 2, 4, 8, 16]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for idx, t_val in enumerate(times):
    ax = axes.flat[idx]

    # Vy = cos(kz * z) * cos(omega(x) * t) where omega(x) = CA(x) * kz
    omega_x = CA_profile * kz
    Vy = np.cos(kz * z) * np.cos(omega_x * t_val)

    ax.plot(x, Vy, "b-", lw=1.5)
    ax.set_title(f"t = {t_val}", fontsize=11)
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel("x (transverse)", fontsize=10)
    ax.set_ylabel("$V_y$", fontsize=10)

    # Show Alfvén speed profile
    ax2 = ax.twinx()
    ax2.plot(x, CA_profile, "r--", alpha=0.3, lw=1)
    ax2.set_ylabel("$C_A(x)$", fontsize=9, color="red", alpha=0.5)
    ax2.set_ylim(0, 2)

fig.suptitle("Alfvén Wave Phase Mixing / Alfvén 파동 위상 혼합\n"
             "As time progresses, transverse gradients steepen dramatically\n"
             "시간이 지남에 따라 횡방향 경사가 급격히 증가",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# --- Decay comparison: homogeneous vs phase-mixed ---
fig, ax = plt.subplots(figsize=(10, 5))

nu = 0.01  # viscosity coefficient
kz_val = 2 * np.pi
dCA_dx_values = [0, 0.5, 1.0, 2.0]
t_arr = np.linspace(0, 4, 500)

for dCA, ls, label in zip(dCA_dx_values,
                           ["-", ":", "--", "-."],
                           ["dCₐ/dx = 0 (homogeneous)",
                            "dCₐ/dx = 0.5", "dCₐ/dx = 1.0", "dCₐ/dx = 2.0"]):
    if dCA == 0:
        # Homogeneous: exponential decay with timescale ~ Re
        decay = np.exp(-nu * kz_val**2 * t_arr)
    else:
        # Phase-mixed: decay ~ exp(-nu * kz^2 * (dCA/dx)^2 * t^3 / 6)
        decay = np.exp(-nu * kz_val**2 * dCA**2 * t_arr**3 / 6)

    ax.plot(t_arr, decay, ls=ls, lw=2, label=label)

ax.set_xlabel("Time (arb. units)", fontsize=12)
ax.set_ylabel("Wave amplitude", fontsize=12)
ax.set_title("Phase Mixing Enhanced Dissipation (cf. Figure 7)\n"
             "위상 혼합에 의한 강화된 산일", fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## Part 6: Propagating Slow Waves — Burgers Equation
## 파트 6: 전파하는 느린 파동 — Burgers 방정식

확장된 Burgers 방정식을 수치적으로 풀어 코로나 루프에서 전파하는 느린 파동의 진화를 시뮬레이션합니다.
We numerically solve the extended Burgers equation to simulate the evolution of propagating slow waves in coronal loops.

$$\frac{\partial A}{\partial s} - a_1 A - a_2\frac{\partial^2 A}{\partial \xi^2} + a_3 A\frac{\partial A}{\partial \xi} = 0$$

In [ ]:
def solve_burgers(a1, a2, a3, xi, ds, n_steps, A_init):
    """Solve the extended Burgers equation using finite differences.

    dA/ds = a1*A + a2*d^2A/dxi^2 - a3*A*dA/dxi

    Args:
        a1: Stratification coefficient (amplification).
        a2: Dissipation coefficient.
        a3: Nonlinearity coefficient.
        xi: Spatial coordinate array (running coordinate).
        ds: Step size along the loop.
        n_steps: Number of steps.
        A_init: Initial amplitude profile A(xi, s=0).

    Returns:
        A_final: Amplitude at final step.
        A_history: List of amplitude snapshots.
    """
    dxi = xi[1] - xi[0]
    A = A_init.copy()
    history = [A.copy()]
    save_every = max(1, n_steps // 10)

    for step in range(n_steps):
        # Second derivative (central difference)
        d2A = np.zeros_like(A)
        d2A[1:-1] = (A[2:] - 2 * A[1:-1] + A[:-2]) / dxi**2

        # First derivative (upwind for stability)
        dA = np.zeros_like(A)
        dA[1:-1] = (A[2:] - A[:-2]) / (2 * dxi)

        # Update
        A = A + ds * (a1 * A + a2 * d2A - a3 * A * dA)

        # Boundary conditions: zero at edges
        A[0] = 0
        A[-1] = 0

        if (step + 1) % save_every == 0:
            history.append(A.copy())

    return A, history


# --- Simulation ---
N_xi = 500
xi = np.linspace(-20, 20, N_xi)

# Initial condition: Gaussian pulse
A_init = 0.05 * np.exp(-xi**2 / 4)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Case 1: Linear wave with dissipation only (a1=0, a3=0)
_, hist1 = solve_burgers(a1=0, a2=0.1, a3=0, xi=xi, ds=0.01, n_steps=500,
                          A_init=A_init)
ax = axes[0]
for i, snap in enumerate(hist1):
    alpha = 0.3 + 0.7 * i / len(hist1)
    ax.plot(xi, snap, alpha=alpha, color=plt.cm.Blues(alpha))
ax.set_title("Linear + Dissipation Only\n선형 + 산일만 ($a_1=0, a_3=0$)", fontsize=11)
ax.set_xlabel(r"$\xi$ (running coordinate)", fontsize=11)
ax.set_ylabel("Amplitude A", fontsize=11)

# Case 2: With stratification (amplification)
_, hist2 = solve_burgers(a1=0.3, a2=0.1, a3=0, xi=xi, ds=0.01, n_steps=300,
                          A_init=A_init)
ax = axes[1]
for i, snap in enumerate(hist2):
    alpha = 0.3 + 0.7 * i / len(hist2)
    ax.plot(xi, snap, alpha=alpha, color=plt.cm.Reds(alpha))
ax.set_title("Linear + Stratification + Dissipation\n"
             "선형 + 성층화 + 산일 ($a_1>0$)", fontsize=11)
ax.set_xlabel(r"$\xi$ (running coordinate)", fontsize=11)
ax.set_ylabel("Amplitude A", fontsize=11)

# Case 3: Full nonlinear Burgers (stratification + dissipation + nonlinearity)
_, hist3 = solve_burgers(a1=0.3, a2=0.1, a3=2.0, xi=xi, ds=0.005, n_steps=600,
                          A_init=A_init)
ax = axes[2]
for i, snap in enumerate(hist3):
    alpha = 0.3 + 0.7 * i / len(hist3)
    ax.plot(xi, snap, alpha=alpha, color=plt.cm.Greens(alpha))
ax.set_title("Full Nonlinear Burgers\n완전 비선형 Burgers ($a_1, a_2, a_3 > 0$)",
             fontsize=11)
ax.set_xlabel(r"$\xi$ (running coordinate)", fontsize=11)
ax.set_ylabel("Amplitude A", fontsize=11)

fig.suptitle("Evolution of Propagating Slow Waves (Eq. 52)\n"
             "전파하는 느린 파동의 진화 (darker = further along loop)\n"
             "(진한 색 = 루프를 따라 더 진행)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Part 7: Fast Wave Train Dispersive Evolution & "Tadpole" Signature
## 파트 7: Fast Wave Train 분산 진화와 "Tadpole" 시그니처

충격적으로 생성된 fast magnetoacoustic 펄스가 분산에 의해 준주기 파열로 진화하는 과정을 시뮬레이션합니다 (cf. Figure 21).
We simulate the dispersive evolution of an impulsively generated fast magnetoacoustic pulse into a quasi-periodic wave train (cf. Figure 21).

Epstein 프로파일 슬랩에서 sausage 모드의 군속도가 파수에 따라 변하므로, 광대역 펄스의 각 Fourier 성분이 서로 다른 속도로 전파하여 분산 파열이 형성됩니다.
In a slab with Epstein profile, the group speed of the sausage mode varies with wavenumber, so each Fourier component of a broadband pulse propagates at different speeds, forming a dispersive wave train.

In [ ]:
# --- Dispersive evolution of fast wave train ---
# Simple model: dispersion relation omega(k) with group speed minimum
# This creates the characteristic "tadpole" wavelet signature

# Model dispersion: omega = k * Vph(k)
# For sausage mode, Vph decreases from CAe to CA0 as k increases
# Use a simple model: Vph(k) = CA0 + (CAe - CA0) / (1 + (k*a)^2)
# Group speed: Vg = d(omega)/dk

CA0_model = 1.0  # Internal Alfvén speed (normalized)
CAe_model = 2.0  # External Alfvén speed
a_model = 1.0    # Slab half-width

k_arr = np.linspace(0.1, 15, 1000)

# Phase speed
Vph = CA0_model + (CAe_model - CA0_model) / (1 + (k_arr * a_model)**2)

# Angular frequency
omega_arr = k_arr * Vph

# Group speed: d omega / dk (numerical)
Vg = np.gradient(omega_arr, k_arr)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Dispersion relation
ax = axes[0]
ax.plot(k_arr * a_model, Vph, "b-", lw=2, label="Phase speed $V_{ph}$")
ax.plot(k_arr * a_model, Vg, "r--", lw=2, label="Group speed $V_g$")
ax.axhline(y=CA0_model, color="gray", ls=":", alpha=0.5, label="$C_{A0}$")
ax.axhline(y=CAe_model, color="gray", ls="--", alpha=0.5, label="$C_{Ae}$")
ax.set_xlabel(r"$k_z a$", fontsize=12)
ax.set_ylabel("Speed", fontsize=12)
ax.set_title("Dispersion of Sausage Mode\n소시지 모드의 분산", fontsize=11)
ax.legend(fontsize=9)
ax.set_xlim(0, 10)
ax.set_ylim(0, 2.5)

# Panel 2: Dispersive wave train at a fixed observation point
# Each Fourier component arrives at different times
z_obs = 70 * a_model  # Observation distance from source

# Arrival time for each k: t(k) = z_obs / Vg(k)
t_arrival = z_obs / Vg

# Signal: superposition of wave components arriving at different times
t_signal = np.linspace(30, 110, 2000)
signal = np.zeros_like(t_signal)

for i in range(len(k_arr)):
    if Vg[i] > 0.1:  # Only propagating components
        # Each component contributes at its arrival time
        width = 0.5
        envelope = np.exp(-0.5 * ((t_signal - t_arrival[i]) / width)**2)
        signal += envelope * np.cos(omega_arr[i] * t_signal - k_arr[i] * z_obs) * 0.01

# Normalize
signal = signal / np.max(np.abs(signal)) * 0.02

ax = axes[1]
ax.plot(t_signal, signal, "b-", lw=1)
ax.set_xlabel("Time (arb. units)", fontsize=12)
ax.set_ylabel("Signal amplitude", fontsize=12)
ax.set_title("Dispersive Wave Train at $z = 70a$\n"
             "분산 파열 ($z = 70a$에서 관측)", fontsize=11)

# Panel 3: Wavelet transform (crude CWT approximation)
ax = axes[2]

# Simple sliding-window spectral analysis
n_periods = 50
periods = np.linspace(1, 15, n_periods)
freqs = 1.0 / periods

power = np.zeros((n_periods, len(t_signal)))

for ip, period in enumerate(periods):
    # Morlet-like wavelet
    omega_w = 2 * np.pi / period
    for it, t0 in enumerate(t_signal):
        window = np.exp(-0.5 * ((t_signal - t0) / (period * 1.5))**2)
        power[ip, it] = np.abs(np.sum(signal * window *
                                       np.exp(-1j * omega_w * t_signal)))**2

# Normalize each frequency
for ip in range(n_periods):
    max_val = np.max(power[ip])
    if max_val > 0:
        power[ip] /= max_val

ax.pcolormesh(t_signal, periods, power, cmap="hot_r", shading="auto")
ax.set_xlabel("Time (arb. units)", fontsize=12)
ax.set_ylabel("Period (arb. units)", fontsize=12)
ax.set_title('"Tadpole" Wavelet Signature (cf. Fig 21)\n'
             '"올챙이" 웨이블릿 시그니처', fontsize=11)
ax.set_ylim(1, 15)

plt.tight_layout()
plt.show()

## Summary / 요약

| Part | 구현 내용 / Implementation | 핵심 결과 / Key Result |
|------|--------------------------|---------------------|
| 1 | Edwin & Roberts 분산 다이어그램 재현 | Fast/slow 밴드, kink/sausage 모드 확인 |
| 2 | Kink 진동 → 코로나 자기장 추정 | B₀ ≈ 13 G (n₀ = 3×10⁹ cm⁻³) |
| 3 | 감쇠 메커니즘 스케일링 비교 | 공진 흡수(τ ∝ P)가 관측에 가장 일치 |
| 4 | Sausage 모드 존재 조건 | 플레어 루프에서만 가능 (L/2a < 0.65√(ρ₀/ρₑ)) |
| 5 | Alfvén 파동 위상 혼합 | 횡방향 경사 급증 → Re^(1/3) 스케일링 감쇠 |
| 6 | Burgers 방정식으로 느린 파동 모델링 | 성층화·산일·비선형 경쟁의 효과 시각화 |
| 7 | Fast wave train 분산 진화 | "Tadpole" 웨이블릿 시그니처 재현 |

**다음 논문 / Next paper**: LRSP #4 — Sheeley (2005), "Surface Evolution of the Sun's Magnetic Field: A Historical Review of the Flux-Transport Mechanism"